In [1]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
)


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import (
    ALL_CHAINS,
    ROOT_PRICE_ORACLE,
    ChainData,
    STATS_CALCULATOR_REGISTRY,
    WETH,
    ETH_CHAIN,
)

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks,
)


chain = ETH_CHAIN

possible_blocks = build_blocks_to_use(chain)

missing_blocks = get_subset_not_already_in_column(
    AutopoolDestinationStates,
    AutopoolDestinationStates.block,
    possible_blocks,
    where_clause=AutopoolDestinationStates.chain_id == chain.chain_id,
)


autopool_df = get_full_table_as_df(Autopools, where_clause=Autopools.chain_id == chain.chain_id)
full_destination_df = natural_left_right_using_where(
    DestinationTokens,
    Destinations,
    using=[DestinationTokens.destination_vault_address, DestinationTokens.chain_id],
    where_clause=DestinationTokens.chain_id == chain.chain_id,
)

token_value_df = natural_left_right_using_where(
    DestinationTokenValues,
    TokenValues,
    using=[DestinationTokenValues.block, DestinationTokens.chain_id, DestinationTokens.token_address],
    where_clause=DestinationTokenValues.chain_id == chain.chain_id,
)
token_df = get_full_table_as_df(Tokens, where_clause=Tokens.chain_id == chain.chain_id)
token_value_df = pd.merge(
    token_value_df, token_df[["symbol", "decimals", "token_address"]], how="left", on="token_address"
)
token_value_df

2025-04-26 17:55:00,112 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-04-26 17:55:00,112 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-26 17:55:00,299 INFO sqlalchemy.engine.Engine select current_schema()
2025-04-26 17:55:00,299 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-26 17:55:00,485 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-04-26 17:55:00,485 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-26 17:55:00,675 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-26 17:55:00,686 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-26 17:55:00,686 INFO sqlalchemy.engine.Engine [generated in 0.00049s] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'dat

,block,chain_id,token_address,destination_vault_address,spot_price,quantity,safe_spot_spread,spot_backing_discount,denomiated_in,backing,safe_price,safe_backing_spread,symbol,decimals
0,20759464,1,0x04C154b66CB340F3Ae24111CC767e0184Ed00Cc6,0x8cA2201BC34780f14Bca452913ecAc8e9928d4cA,0.998460,2.388944e+03,0.000865,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.997598,NaN,pxETH,18
1,20759464,1,0x04C154b66CB340F3Ae24111CC767e0184Ed00Cc6,0xE382BBd32C4E202185762eA433278f4ED9E6151E,0.998586,1.879035e+03,0.000991,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.997598,NaN,pxETH,18
2,20759464,1,0x04C154b66CB340F3Ae24111CC767e0184Ed00Cc6,0xbA1462f43c6f60ebD1C62735c94E428aD073E01A,0.998586,NaN,0.000991,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.997598,NaN,pxETH,18
3,20759464,1,0x04C154b66CB340F3Ae24111CC767e0184Ed00Cc6,0xc4Eb861e7b66f593482a3D7E8adc314f6eEDA30B,0.998460,NaN,0.000865,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.997598,NaN,pxETH,18
4,20759464,1,0x04C154b66CB340F3Ae24111CC767e0184Ed00Cc6,0xd96E943098B2AE81155e98D7DC8BeaB34C539f01,0.998796,2.606154e+03,0.001201,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.997598,NaN,pxETH,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35835,22356658,1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x7876F91BB22148345b3De16af9448081E9853830,0.000550,2.047063e+08,-0.000006,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000550,NaN,USDC,6
35836,22356658,1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0x9Fe20B75709403723610Fd2995D0f4A9e19C3514,0.000550,9.009306e+06,-0.000061,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000550,NaN,USDC,6
35837,22356658,1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xCADe3658eCEA14e4da22ef81BE335E9783E68848,0.000550,4.517452e+05,0.000371,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000550,NaN,USDC,6
35838,22356658,1,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0.000550,9.009306e+06,-0.000061,NaN,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,NaN,0.000550,NaN,USDC,6


In [8]:
autopool_to_all_ever_active_destinations = fetch_autopool_to_active_destinations_over_this_period_of_missing_blocks(
    chain, missing_blocks
)
autopool_to_all_ever_active_destinations

2025-04-26 17:57:34,366 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-26 17:57:34,367 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destinations
            WHERE destinations.chain_id = 1
        
2025-04-26 17:57:34,368 INFO sqlalchemy.engine.Engine [cached since 126.4s ago] {}
2025-04-26 17:57:34,643 INFO sqlalchemy.engine.Engine COMMIT
2025-04-26 17:57:34,737 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-26 17:57:34,738 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopools
            WHERE autopools.chain_id = 1
        
2025-04-26 17:57:34,738 INFO sqlalchemy.engine.Engine [cached since 153.4s ago] {}
2025-04-26 17:57:34,921 INFO sqlalchemy.engine.Engine COMMIT


{'0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56': [Destinations(destination_vault_address='0xf3ae3c74EaD129e770A864CeE291A805b170bBe0', chain_id=1, exchange_name='balancer', block_deployed=20756405, name='Tokemak-Wrapped Ether-Balancer rETH Stable Pool', symbol='toke-WETH-B-rETH-STABLE', pool_type='balMetaStable', pool='0x1E19CF2D73a72Ef1332C882F20534B6519Be0276', underlying='0x1E19CF2D73a72Ef1332C882F20534B6519Be0276', underlying_symbol='B-rETH-STABLE', underlying_name='B-rETH-STABLE'),
  Destinations(destination_vault_address='0x865e59D439BF7310c9BC6117E6020B8C87De4065', chain_id=1, exchange_name='balancer', block_deployed=20756406, name='Tokemak-Wrapped Ether-Balancer osETH/wETH StablePool', symbol='toke-WETH-osETH/wETH-BPT', pool_type='balCompStable', pool='0xDACf5Fa19b1f720111609043ac67A9818262850c', underlying='0xDACf5Fa19b1f720111609043ac67A9818262850c', underlying_symbol='osETH/wETH-BPT', underlying_name='osETH/wETH-BPT'),
  Destinations(destination_vault_address='0x25cb41919d6B88

In [32]:
def build_autopool_balance_of_calls_by_destination(
    autopool_vault_address: str, destination_vault_addresses: list[str]
) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["balanceOf(address)(uint256)", autopool_vault_address],
            [((autopool_vault_address, destination_vault_address), safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


# DestinationVaultAddress.underlyingTotalSupply()
#  -> I'm 95% sure this is the total supply of the lp tokens, not the quantity of lp tokens staked for rewards
def build_destinations_underlyingTotalSupply_calls(destination_vault_addresses: list[str]) -> list[Call]:
    return [
        Call(
            destination_vault_address,
            ["underlyingTotalSupply()(uint256)"],
            [(destination_vault_address, safe_normalize_with_bool_success)],
        )
        for destination_vault_address in destination_vault_addresses
    ]


all_active_destinations = set()

for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
    this_autopool_active_destinations = [
        dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
    ]
    all_active_destinations.update(this_autopool_active_destinations)

autopool_balance_of_calls = []

for autopool_vault_address in autopool_to_all_ever_active_destinations.keys():
    this_autopool_active_destinations = [
        dest.destination_vault_address for dest in autopool_to_all_ever_active_destinations[autopool_vault_address]
    ]

    autopool_balance_of_calls.extend(
        build_autopool_balance_of_calls_by_destination(autopool_vault_address, this_autopool_active_destinations)
    )

calls = [*build_destinations_underlyingTotalSupply_calls(all_active_destinations), *autopool_balance_of_calls]

autopool_destination_balance_of_df = get_raw_state_by_blocks(calls, missing_blocks, chain, include_block_number=True)
autopool_destination_balance_of_df

[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]
[Errno 1] [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2578) [0]


,0x2B08137BeABd2454AD3631DEB754F97C5c93eB78,0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04,0x5A4B544B9734930DDC587c9a2f093dC5058A4f4D,0xdfE3fA7027E84f59b266459C567278C79fe86f0C,0x148Ca723BefeA7b021C399413b8b7426A4701500,0xE382BBd32C4E202185762eA433278f4ED9E6151E,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2,0xd96E943098B2AE81155e98D7DC8BeaB34C539f01,0x356c79Ab2B2cEFAB685004CE827146058A6c3e77,0x6a8C6ff78082a2ae494EB9291DdC7254117298Ff,...,"(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x2f2CC1bf461413014741dD68481dB4a3686DAC3D)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x40219bBda953ca811d2D0168Dc806a96b84791d9)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0xC9b5D82652a1C8214b0971A004983d0EEeDD751C)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0xf9779aEF9f77e78C857CB4A068c65CcBee25BAAc)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x60339056EC88996e41757E05a798310E46972cca)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, 0x5c6aeb9ef0d5BbA4E6691f381003503FD0D45126)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xc4Eb861e7b66f593482a3D7E8adc314f6eEDA30B)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, 0xbA1462f43c6f60ebD1C62735c94E428aD073E01A)",block
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.248990,3004.943853,352.135269,4418.771443,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20759464
2024-09-16 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.306071,3024.939003,352.151418,4418.771443,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20766617
2024-09-17 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.375275,3124.928602,484.989135,4414.993123,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20773761
2024-09-18 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.571438,3389.359227,485.059853,4561.283097,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20780916
2024-09-19 23:59:59+00:00,NaN,NaN,NaN,NaN,2550.672146,3724.974425,485.091132,4598.273271,NaN,84.650070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20788084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-22 23:59:59+00:00,227.502467,8023.442403,12327.821057,684.921261,1628.949736,6199.358769,0.784321,8023.442403,5942.370219,227.502467,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22327989
2025-04-23 23:59:59+00:00,227.502467,8078.410498,12328.542843,688.107559,1629.680081,6029.808420,0.784354,8078.410498,5942.370219,227.502467,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22335153
2025-04-24 23:59:59+00:00,264.159941,8052.117527,11642.586887,734.304846,1629.830873,6079.772900,0.784384,8052.117527,5942.370219,264.159941,...,0.0,0.0,0.0,0.0,0.0,470.519208,317.661117,0.0,540.652839,22342307


In [ ]:
AutopoolDestinationStates_new_partial_rows = []

def _to_apply_over_each_row(row:dict):
    
    for autopool_dest_tuple, balance_of in row.items():
        if autopool_dest_tuple != 'block':
            autopool_vault_address, destination_vault_address = col # unpack the tuple

            AutopoolDestinationStates_new_partial_rows.append(
                        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':destination_vault_address, 
            'block': row['block'],
            'amount': balance_of,
            # this will raise an error if trying to insert into AutopoolDestinationStates because nullable = false 
            'total_safe_value': None,
            'total_spot_value': None,
            'total_backing_value': None,
            'percent_ownership': -100.0, # depends on destination_states.underlying_token_total_supply
        }

            )

# t



for col in autopool_destination_balance_of_df.columns:
    this_autopool_destination_balance_of = []
    autopool_balance_columns = autopool_destination_balance_of_df[col]
    autopool_vault_address, destination_vault_address = col # unpack
    this_autopool_destination_balance_of.append(
        {
            'autopool_vault_address':autopool_vault_address,
            'destination_vault_address':

        }

    )



0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xf3ae3c74EaD129e770A864CeE291A805b170bBe0
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x865e59D439BF7310c9BC6117E6020B8C87De4065
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xfB6f99FdF12E37Bfe3c4Cf81067faB10c465fb24
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x8cA2201BC34780f14Bca452913ecAc8e9928d4cA
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x896eCc16Ab4AFfF6cE0765A5B924BaECd7Fa455a
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xC001f23397dB71B17602Ce7D90a983Edc38DB0d1
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0x6a8C6ff78082a2ae494EB9291DdC7254117298Ff
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xd96E943098B2AE81155e98D7DC8BeaB34C539f01
0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56_0xE382BBd32

In [6]:
autopool_to_all_ever_active_destinations.keys()

dict_keys(['0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', '0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5', '0xE800e3760FC20aA98c5df6A9816147f190455AF3', '0x35911af1B570E26f668905595dEd133D01CD3E5a'])

In [ ]:
autopool_to_all_ever_active_destinations.keys

In [3]:
# def build_pool_token_spot_price_calls(
#     chain: ChainData, pool_addresses: list[str], token_addresses: list[str]
# ) -> list[Call]:

#     return [
#         Call(
#             ROOT_PRICE_ORACLE(chain),
#             ["getSpotPriceInEth(address,address)(uint256)", token_address, pool_address],
#             [(f"{pool_address}_{token_address}", safe_normalize_with_bool_success)],
#         )
#         for (pool_address, token_address) in zip(pool_addresses, token_addresses)
#     ]


# spot_price_calls = build_pool_token_spot_price_calls(
#     chain, full_destination_df["pool"], full_destination_df["token_address"]
# )
# destination_token_spot_price_df = get_raw_state_by_blocks(
#     spot_price_calls, possible_blocks, chain, include_block_number=True
# )
# destination_token_spot_price_df


# def build_underlying_reserves_calls(destinations: list[str]) -> list[Call]:
#     return [
#         Call(
#             dest,
#             "underlyingReserves()(address[],uint256[])",
#             [
#                 (f"{dest}_underlyingReserves_tokens", identity_with_bool_success),
#                 (f"{dest}_underlyingReserves_amounts", identity_with_bool_success),
#             ],
#         )
#         for dest in destinations
#     ]


# underlying_reserves_calls = build_underlying_reserves_calls(full_destination_df["destination_vault_address"])
# underlying_reserves_df = get_raw_state_by_blocks(
#     underlying_reserves_calls, possible_blocks, chain, include_block_number=True
# )
# underlying_reserves_df